In [1]:
import sys
sys.path.append("/cellarold/users/mpagadal/Programs/anaconda3/lib/python3.7/site-packages")

In [2]:
import pandas as pd
import os
import json
import numpy as np

In [3]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [4]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib_venn import venn3

In [5]:
def make_circos(df):
    df["chr"]="hs"+df["#CHROM"].astype(str)
    df["bp1"]=df["POS"]
    df["bp2"]=df["POS"]+1
    df["p"]=-np.log10(df["P"])
    return(df[["chr","bp1","bp2","p"]])

In [6]:
def compile_statistics(prefix,direct):
    compiled_df=pd.DataFrame()
    files=[x for x in os.listdir(direct) if prefix in x]
    print("compiling {} files".format(len(files)))
    for x in files:
        df=pd.read_csv(direct+x,delimiter="\t")
        compiled_df=compiled_df.append(df)
    return(compiled_df)
    

In [7]:
def Merge(dict1, dict2):
    '''
    Function to merge python dictionaries
    '''
    res = {**dict1, **dict2}
    return res

In [8]:
def rsid_map(df,rsid):
    '''
    Function to map to rsid using hg19 rsid map, taking into account allele
    '''
    
    df["snp_noallele"]=df["ID"].str.rsplit(":",2).str[0]
    rsid_filt=rsid[rsid["variant"].isin(df["snp_noallele"].tolist())]
    
    rsid_filt["snp1"]=rsid_filt[0].astype(str)+":"+rsid_filt[1].astype(str)+":"+rsid_filt[3]+":"+rsid_filt[4]
    rsid_filt["snp2"]=rsid_filt[0].astype(str)+":"+rsid_filt[1].astype(str)+":"+rsid_filt[4]+":"+rsid_filt[3]
    
    mp_snp1=dict(zip(rsid_filt["snp1"],rsid_filt[5]))
    mp_snp2=dict(zip(rsid_filt["snp2"],rsid_filt[5]))
    
    mp_rsid=Merge(mp_snp1,mp_snp2)
    
    df["rsid"]=df["ID"].map(mp_rsid)
    df=df[~df["rsid"].isnull()]
    df["ID"]=df["rsid"]
    del df["rsid"]
    return(df)

In [9]:
def process_phewas(df):
    df=df[~df["p"].isnull()]
    df=df[df["p"]!="p"]
    df["p"]=df["p"].astype(float)
    
    df["snp"]=df["snp"].str.replace("`","")
    df["description"]=df["description"].str.replace(":","")
    df["description"]=df["description"].str.replace(";","")
    
    return(df)

In [10]:
def process_labwas(df):
    df["pheno"]=df["file"].str.split(".dose.").str[1]
    df["pheno"]=df["pheno"].str.split(".glm.linear").str[0]
    return(df)

In [11]:
def format4vep(snps,col):
    vep=pd.DataFrame({"snps":snps[col]})
    vep=vep["snps"].str.split(":",expand=True)
    vep[3]=vep[3].astype(str)+"/"+vep[2].astype(str)
    vep[2]=vep[1]
    vep[4]="+"
    vep[5]=snps[col]
    vep[0]=vep[0].str.replace({"X","23"},regex=True)
    print(vep)
    vep[0]=pd.to_numeric(vep[0])
    vep[1]=pd.to_numeric(vep[1])
    vep[2]=pd.to_numeric(vep[2])
    
    vep=vep.sort_values(by=[0, 2])
    vep[0]=vep[0].astype(str)
    return(vep)
    

### Compile summary statistics for European, African and Hispanic ancestral groups

mvp data was exported in chunks and needs to be compiled

In [40]:
# eur=compile_statistics("eur","/cellar/users/mpagadal/projects/TestosteroneGWAS/discovery/data/summarystats/chunks/")
# afr=compile_statistics("afr","/cellar/users/mpagadal/projects/TestosteroneGWAS/discovery/data/summarystats/chunks/")
# his=compile_statistics("his","/cellar/users/mpagadal/projects/TestosteroneGWAS/discovery/data/summarystats/chunks/")

# eur.to_csv("../data/summarystats/full/compiled.eur.release4.testosterone.glm.linear",index=None,sep="\t")
# afr.to_csv("../data/summarystats/full/compiled.afr.release4.testosterone.glm.linear",index=None,sep="\t")
# his.to_csv("../data/summarystats/full/compiled.his.release4.testosterone.glm.linear",index=None,sep="\t")

# # rsid annotation

# rsid=pd.read_csv("/cellar/users/mpagadal/resources/rsid/hg19_avsnp147.txt",header=None,delimiter="\t")
# rsid["variant"]=rsid[0].astype(str)+":"+rsid[1].astype(str)

# eur_rsid=rsid_map(eur,rsid)
# afr_rsid=rsid_map(afr,rsid)
# his_rsid=rsid_map(his,rsid)

# eur_rsid["BETA"]=np.where(eur_rsid["A1"]!=eur_rsid["ALT"],eur_rsid["BETA"]*-1,eur_rsid["BETA"])
# afr_rsid["BETA"]=np.where(afr_rsid["A1"]!=afr_rsid["ALT"],afr_rsid["BETA"]*-1,afr_rsid["BETA"])
# his_rsid["BETA"]=np.where(his_rsid["A1"]!=his_rsid["ALT"],his_rsid["BETA"]*-1,his_rsid["BETA"])

# cols=["#CHROM","POS","ID","REF","ALT","TEST","OBS_CT","BETA","SE", "T_STAT","P"]

# eur_rsid[cols].to_csv("../data/summarystats/full/compiled.eur.rsid.release4.testosterone.glm.linear",index=None,sep="\t")
# afr_rsid[cols].to_csv("../data/summarystats/full/compiled.afr.rsid.release4.testosterone.glm.linear",index=None,sep="\t")
# his_rsid[cols].to_csv("../data/summarystats/full/compiled.his.rsid.release4.testosterone.glm.linear",index=None,sep="\t")

### load summary statistics

In [41]:
eur=pd.read_csv("../data/summarystats/full/compiled.eur.release4.testosterone.glm.linear",sep="\t")
afr=pd.read_csv("../data/summarystats/full/compiled.afr.release4.testosterone.glm.linear",sep="\t")
his=pd.read_csv("../data/summarystats/full/compiled.his.release4.testosterone.glm.linear",sep="\t")

eur_rsid=pd.read_csv("../data/summarystats/full/compiled.eur.rsid.release4.testosterone.glm.linear",sep="\t")
afr_rsid=pd.read_csv("../data/summarystats/full/compiled.afr.rsid.release4.testosterone.glm.linear",sep="\t")
his_rsid=pd.read_csv("../data/summarystats/full/compiled.his.rsid.release4.testosterone.glm.linear",sep="\t")

##### get significant clumps

In [42]:
supptable1=pd.DataFrame()

for x in os.listdir("../data/clumps/"):
    df=pd.read_csv("../data/clumps/"+x,header=None)
    df=df.rename(columns={0:"MVP_variant"})
    df["group"]=x.split(".")[1]
    supptable1=supptable1.append(df)

##### map to rsid

In [43]:
supptable1["MVP_variant"]=supptable1["MVP_variant"].replace({"2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT":"2:31989359:G:A"})

eur_rsid["MVP_variant"]=eur_rsid["#CHROM"].astype(str)+":"+eur_rsid["POS"].astype(str)+":"+eur_rsid["REF"]+":"+eur_rsid["ALT"]
afr_rsid["MVP_variant"]=afr_rsid["#CHROM"].astype(str)+":"+afr_rsid["POS"].astype(str)+":"+afr_rsid["REF"]+":"+afr_rsid["ALT"]
his_rsid["MVP_variant"]=his_rsid["#CHROM"].astype(str)+":"+his_rsid["POS"].astype(str)+":"+his_rsid["REF"]+":"+his_rsid["ALT"]

eur_rsid_filt=eur_rsid[eur_rsid["MVP_variant"].isin(supptable1["MVP_variant"].tolist())]
afr_rsid_filt=afr_rsid[afr_rsid["MVP_variant"].isin(supptable1["MVP_variant"].tolist())]
his_rsid_filt=his_rsid[his_rsid["MVP_variant"].isin(supptable1["MVP_variant"].tolist())]

rsid_dict=Merge(dict(zip(eur_rsid_filt["MVP_variant"],eur_rsid_filt["ID"])),dict(zip(afr_rsid_filt["MVP_variant"],afr_rsid_filt["ID"])))
rsid_dict=Merge(rsid_dict,dict(zip(his_rsid_filt["MVP_variant"],his_rsid_filt["ID"])))
rsid_dict["7:9714084:G:A"]="rs1042554135"

supptable1["rsid"]=supptable1["MVP_variant"].map(rsid_dict)
supptable1["rsid"].to_csv("ldtrait.txt",header=None,index=None)


/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:16: FutureWarning: The signature of `Series.to_csv` was aligned to that of `DataFrame.to_csv`, and argument 'header' will change its default value from False to True: please pass an explicit value to suppress this warning.
  app.launch_new_instance()


##### map beta, p-value and A1 allele

In [44]:
eur_summary=pd.read_csv("../data/summarystats/significant/compiled.eur.all.variant.glm.linear",delimiter="\t")
eur_summary=eur_summary.rename(columns={"ID":"MVP_variant","BETA":"BETA_eur","P":"P_eur","A1":"A1_eur"})
eur_summary["MVP_variant"]=eur_summary["MVP_variant"].replace({"2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT":"2:31989359:G:A"})

afr_summary=pd.read_csv("../data/summarystats/significant/compiled.afr.all.variant.glm.linear",delimiter="\t")
afr_summary=afr_summary.rename(columns={"ID":"MVP_variant","BETA":"BETA_afr","P":"P_afr","A1":"A1_afr"})
afr_summary["MVP_variant"]=afr_summary["MVP_variant"].replace({"2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT":"2:31989359:G:A"})

his_summary=pd.read_csv("../data/summarystats/significant/compiled.his.all.variant.glm.linear",delimiter="\t")
his_summary=his_summary.rename(columns={"ID":"MVP_variant","BETA":"BETA_his","P":"P_his","A1":"A1_his"})
his_summary["MVP_variant"]=his_summary["MVP_variant"].replace({"2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT":"2:31989359:G:A"})

supptable1=pd.merge(supptable1,eur_summary[["MVP_variant","BETA_eur","P_eur","A1_eur"]])
supptable1=pd.merge(supptable1,afr_summary[["MVP_variant","BETA_afr","P_afr","A1_afr"]])
supptable1=pd.merge(supptable1,his_summary[["MVP_variant","BETA_his","P_his","A1_his"]])

# orient to European A1 allele

supptable1["BETA_afr"]=np.where(supptable1["A1_eur"]!=supptable1["A1_afr"],supptable1["BETA_afr"]*-1,supptable1["BETA_afr"])
supptable1["BETA_his"]=np.where(supptable1["A1_eur"]!=supptable1["A1_his"],supptable1["BETA_his"]*-1,supptable1["BETA_his"])

del supptable1["A1_afr"]
del supptable1["A1_his"]
supptable1=supptable1.rename(columns={"A1_eur":"A1"})

##### map metal variants

In [45]:
metal=pd.read_csv("../data/metal/METAANALYSIS1.TBL.filt",delimiter="\t")
metal_snps=metal[metal["P-value"]<.00000005]
metal_snps["MarkerName"]=metal_snps["MarkerName"].replace({"2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT":"2:31989359:G:A"})
supptable1["metal"]=np.where(supptable1["MVP_variant"].isin(metal_snps["MarkerName"].tolist()),"yes","no")

/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [46]:
supptable1["metal"].value_counts()

yes    46
no     17
Name: metal, dtype: int64

##### map frequencies

In [47]:
eur_freq=pd.read_csv("../data/freq/compiled.eur.freq",header=None,sep="\t")
afr_freq=pd.read_csv("../data/freq/compiled.afr.freq",header=None,sep="\t")
his_freq=pd.read_csv("../data/freq/compiled.his.freq",header=None,sep="\t")

eur_freq[1]=eur_freq[1].replace("2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT","2:31989359:G:A")
afr_freq[1]=afr_freq[1].replace("2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT","2:31989359:G:A")
his_freq[1]=his_freq[1].replace("2:31989359:GGAGTATGTTCTTGCTGAT:AGAGTATGTTCTTGCTGAT","2:31989359:G:A")

eur_freq["FREQ_eur"]=eur_freq[4].astype(str)+" ("+eur_freq[3]+")"
afr_freq["FREQ_afr"]=afr_freq[4].astype(str)+" ("+afr_freq[3]+")"
his_freq["FREQ_his"]=his_freq[4].astype(str)+" ("+his_freq[3]+")"

eur_freq=eur_freq.rename(columns={1:"MVP_variant"})
afr_freq=afr_freq.rename(columns={1:"MVP_variant"})
his_freq=his_freq.rename(columns={1:"MVP_variant"})

supptable1=pd.merge(supptable1,eur_freq[["MVP_variant","FREQ_eur"]],on="MVP_variant",how="left")
supptable1=pd.merge(supptable1,afr_freq[["MVP_variant","FREQ_afr"]],on="MVP_variant",how="left")
supptable1=pd.merge(supptable1,his_freq[["MVP_variant","FREQ_his"]],on="MVP_variant",how="left")

##### compare beta values

In [48]:
supptable1["afr_diff"]=np.where(np.sign(supptable1["BETA_eur"])!=np.sign(supptable1["BETA_afr"]),1,0)
supptable1["his_diff"]=np.where(np.sign(supptable1["BETA_eur"])!=np.sign(supptable1["BETA_his"]),1,0)

##### full phewas associations

In [49]:
eur_phewas_sig=pd.read_csv("../data/phewas/eur.phewas.sig.csv")
afr_phewas_sig=pd.read_csv("../data/phewas/afr.phewas.sig.csv")
his_phewas_sig=pd.read_csv("../data/phewas/his.phewas.sig.csv")

eur_phewas_annot=eur_phewas_sig.groupby('snp')['description'].apply(lambda x: ','.join(x)).reset_index()
eur_phewas_annot["snp"]=eur_phewas_annot["snp"].str.split("_").str[0]
dict_eur_phewas=dict(zip(eur_phewas_annot["snp"],eur_phewas_annot["description"]))

afr_phewas_annot=afr_phewas_sig.groupby('snp')['description'].apply(lambda x: ','.join(x)).reset_index()
afr_phewas_annot["snp"]=afr_phewas_annot["snp"].str.split("_").str[0]
dict_afr_phewas=dict(zip(afr_phewas_annot["snp"],afr_phewas_annot["description"]))

his_phewas_annot=his_phewas_sig.groupby('snp')['description'].apply(lambda x: ','.join(x)).reset_index()
his_phewas_annot["snp"]=his_phewas_annot["snp"].str.split("_").str[0]
dict_his_phewas=dict(zip(his_phewas_annot["snp"],his_phewas_annot["description"]))

supptable1["phewas_eur"]=supptable1["MVP_variant"].map(dict_eur_phewas)
supptable1["phewas_afr"]=supptable1["MVP_variant"].map(dict_afr_phewas)
supptable1["phewas_his"]=supptable1["MVP_variant"].map(dict_his_phewas)

##### full labwas associations - only mean lab values

In [50]:
eur_labwas_sig=pd.read_csv("../data/labwas/eur.labwas.sig.csv")
afr_labwas_sig=pd.read_csv("../data/labwas/afr.labwas.sig.csv")
his_labwas_sig=pd.read_csv("../data/labwas/his.labwas.sig.csv")

eur_labwas_annot=eur_labwas_sig.groupby('ID')['pheno'].apply(lambda x: ','.join(x)).reset_index()
dict_eur_labwas=dict(zip(eur_labwas_annot["ID"],eur_labwas_annot["pheno"]))

afr_labwas_annot=afr_labwas_sig.groupby('ID')['pheno'].apply(lambda x: ','.join(x)).reset_index()
dict_afr_labwas=dict(zip(afr_labwas_annot["ID"],afr_labwas_annot["pheno"]))

his_labwas_annot=his_labwas_sig.groupby('ID')['pheno'].apply(lambda x: ','.join(x)).reset_index()
dict_his_labwas=dict(zip(his_labwas_annot["ID"],his_labwas_annot["pheno"]))

supptable1["labwas_eur"]=supptable1["MVP_variant"].map(dict_eur_labwas)
supptable1["labwas_afr"]=supptable1["MVP_variant"].map(dict_afr_labwas)
supptable1["labwas_his"]=supptable1["MVP_variant"].map(dict_his_labwas)

##### format for vep

In [51]:
def format4vep(snps,col):
    vep=pd.DataFrame({"snps":snps[col]})
    vep=vep["snps"].str.split(":",expand=True)
    vep[3]=vep[3].astype(str)+"/"+vep[2].astype(str)
    vep[2]=vep[1]
    vep[4]="+"
    vep[5]=snps[col]
    vep[0]=vep[0].str.replace("X","23")
    vep[0]=pd.to_numeric(vep[0])
    vep[1]=pd.to_numeric(vep[1])
    vep[2]=pd.to_numeric(vep[2])
    vep=vep.sort_values(by=[0, 2])
    vep[0]=vep[0].astype(str)
    vep[0]=vep[0].replace("23","X")
    return(vep)

In [53]:
vep=format4vep(supptable1,"MVP_variant")
vep.to_csv("../data/vep/testosterone4vep.csv",index=None,header=None,sep="\t")

##### annotate with vep

In [55]:
#vep results
results=pd.read_csv("../data/vep/ILbuWbC58lrzXsyJ.txt",delimiter="\t")

vep_annot=results.drop_duplicates(["#Uploaded_variation","SYMBOL"]).groupby('#Uploaded_variation')['SYMBOL'].apply(lambda x: ','.join(x)).reset_index()
vep_annot.columns=["MVP_variant","VEP SYMBOL"]

supptable1=pd.merge(supptable1,vep_annot,on="MVP_variant",how="left")

supptable1["chr"]=supptable1["MVP_variant"].str.split(":").str[0]
supptable1["bp"]=supptable1["MVP_variant"].str.split(":").str[1]
supptable1["bp"]=pd.to_numeric(supptable1["bp"])
supptable1=supptable1.sort_values(by=["chr","bp"])

supptable1.to_csv("../files/Supplemental Table 1.tsv",index=None,sep="\t")

##### make files for circos scatter

In [56]:
circos_eur=make_circos(eur)
circos_afr=make_circos(afr)
circos_his=make_circos(his)

In [59]:
circos_eur[circos_eur["p"]>2].to_csv("../data/circos/compiled.eur.circos",header=None,index=None,sep="\t")
circos_afr[circos_afr["p"]>2].to_csv("../data/circos/compiled.afr.circos",header=None,index=None,sep="\t")
circos_his[circos_his["p"]>2].to_csv("../data/circos/compiled.his.circos",header=None,index=None,sep="\t")

##### make circos inner track

In [60]:
metal_snps=supptable1[supptable1["metal"]=="yes"]
metal_snps["bp2"]=metal_snps["bp"]+1
metal_snps["chr"]="hs"+metal_snps["chr"].astype(str)
metal_circos=metal_snps[["chr","bp","bp2"]]
metal_circos["size"]=5

/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  
/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  This is separate from the ipykernel package so we can avoid doing imports until
/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to 

In [61]:
metal_circos.to_csv("../data/circos/metal.circos.txt",header=None,index=None,sep="\t")

In [62]:
nonmetal_snps=supptable1[supptable1["metal"]=="no"]
nonmetal_snps["bp2"]=nonmetal_snps["bp"]+1
nonmetal_snps["chr"]="hs"+nonmetal_snps["chr"].astype(str)
for x in nonmetal_snps["group"].unique():
    nonmetal_circos=nonmetal_snps[nonmetal_snps["group"]==x]
    nonmetal_circos=nonmetal_circos[["chr","bp","bp2"]]
    nonmetal_circos["size"]=5
    nonmetal_circos[["chr","bp","bp2","size"]].to_csv("../data/circos/"+x+".circos.unique.txt",index=None,header=None,sep="\t")

/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  
/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  This is separate from the ipykernel package so we can avoid doing imports until


##### make cochran track

In [64]:
metal["chr"]=metal["MarkerName"].str.split(":").str[0]
metal["chr"]="hs"+metal["chr"]
metal["bp"]=metal["MarkerName"].str.split(":").str[1]
metal["bp2"]=metal["bp"].astype(int)+1
metal["track"]=5
metal["glyph_size"]="glyph_size="+metal["HetChiSq"].astype(int).astype(str)
metal[["chr","bp","bp2","track","glyph_size"]].to_csv("../data/circos/cochran.txt",header=None,index=None,sep="\t")

##### make gene track

In [69]:
with open('/cellar/users/mpagadal/resources/tcga/ensembl_map.json', 'r') as f:
    ensembl_dict = json.load(f)
ensembl_dict={k.split(".")[0]:v for k,v in ensembl_dict.items()}

# coloc results
coloc=pd.read_csv("../data/coloc/coloc.results.csv")
coloc["gene"]=coloc["file"].str.split(".").str[1]
coloc["cell"]=coloc["file"].str.split(".").str[0]
coloc["gene_name"]=coloc["gene"].map(ensembl_dict)
coloc_sig=coloc[coloc["PP.H4.abf"]>0.8]
coloc_sig=coloc_sig[~coloc_sig["gene_name"].str.contains("\.")]

#TWAS results
twas=pd.read_csv("../data/twas/mvp.testosterone.results.csv")
twas_sig=twas[twas["TWAS.P"]<.000001]
twas_sig=twas_sig[~twas_sig["ID"].str.contains("\.")]

#coloc and TWAS intersection
twas_keep_genes=[]

for x in twas_sig["CHR"].unique():
    twas_chr=twas_sig[twas_sig["CHR"]==x].sort_values(by="-log10p")
    twas_chr=twas_chr[~twas_chr["ID"].duplicated()]
    genes=twas_chr.sort_values(by="-log10p")["ID"].tolist()[0:10]
    print(len(genes))
    twas_keep_genes=twas_keep_genes+genes
    
twas_keep=twas_sig[twas_sig["ID"].isin(twas_keep_genes)]
twas_genes=twas_keep[["ID","PANEL"]]
twas_genes.columns=["Gene","Cell"]
coloc_genes=coloc_sig[["gene_name","cell"]]
coloc_genes.columns=["Gene","Cell"]
cell_genes=twas_genes.append(coloc_genes)
cell_genes=cell_genes.drop_duplicates()
cell_gene_counts=cell_genes["Gene"].value_counts().reset_index()
testis_genes=cell_genes[cell_genes["Cell"]=="Testis"]["Gene"].tolist()
liver_genes=cell_genes[cell_genes["Cell"]=="Liver"]["Gene"].tolist()
both_genes=[x for x in testis_genes if x in liver_genes]

#get gene locations
annot=pd.read_csv("/cellar/users/mpagadal/resources/annotations/gencode.v19.annotation.gff3",comment="#",delimiter="\t",header=None)
annot["gene_name"]=annot[8].str.split("gene_name=").str[1]
annot["gene_name"]=annot["gene_name"].str.split(";").str[0]
annot["gene_type"]=annot[8].str.split("transcript_type=").str[1]
annot["gene_type"]=annot["gene_type"].str.split(";").str[0]
annot=annot[annot["gene_type"]=="protein_coding"]
annot=annot[annot[2]=="gene"]
annot["chr"]=annot[0].str.replace("chr","hs")
annot["bp1"]=annot[3].astype(int)
annot["bp2"]=annot["bp1"]+1

#color genes by tissue specificity
annot_combo=annot[annot["gene_name"].isin(both_genes)]
annot_combo["color"]="color=red"
annot_liver=annot[annot["gene_name"].isin([x for x in liver_genes if x not in both_genes])]
annot_liver["color"]="color=green"
annot_testis=annot[annot["gene_name"].isin([x for x in testis_genes if x not in both_genes])]
annot_testis["color"]="color=purple"
annot_rest=annot[annot["gene_name"].isin([x for x in cell_genes["Gene"].tolist() if x not in both_genes+testis_genes+liver_genes])]
annot_rest["color"]="color=gray"
genes_circos=annot_combo[["chr","bp1","bp2","gene_name","color"]].append(annot_liver[["chr","bp1","bp2","gene_name","color"]])
genes_circos=genes_circos.append(annot_testis[["chr","bp1","bp2","gene_name","color"]])
genes_circos=genes_circos.append(annot_rest[["chr","bp1","bp2","gene_name","color"]])
genes_circos.to_csv("../data/circos/genes.txt",index=None,header=None,sep="\t")


8
10
1
5
3
10
2
6
4
6
10
3
2
1
2


/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
/cellar/users/mpagadal/Programs/miniconda3/envs/baseold/lib/python3.7/site-packages/ipykernel_launcher.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index